# sales_intelligence_demo — Pipeline & RAG Walkthrough

The reference example for natural-language queries: a full bronze → silver → gold
pipeline plus **cerebrum** (LLM → SQL), **neuron** (HTTP server), **cortex** (chat UI),
and the full **RAG accuracy** layer — curated `metadata.yaml`/`relationships.yaml`,
dynamic few-shot retrieval, and confidence-gated schema pruning.

| Layer | What happens |
|-------|--------------|
| Bronze | `seed.py` writes `transactions`, `reps`, `targets` CSVs → Parquet |
| Silver | cast + a derived UDF join (`build_rep_performance`) with attainment % |
| Gold | `regional_summary`, `rep_leaderboard`, `quarterly_trend` aggregations |
| cerebrum | NL question → DuckDB SQL → Polars result, all local |
| RAG | curated `metadata.yaml` + `relationships.yaml` + verified examples sharpen accuracy |

See [`README.md`](../../README.md) for the full data model / pipeline flow diagrams.

## Setup — navigate to example root

In [1]:
import os
import polars as pl
from pathlib import Path

# ipynb/ → sales_intel/ → sales_intelligence_demo/
example_root = Path(os.getcwd()).parent.parent
os.chdir(example_root)
print(f'Working directory: {os.getcwd()}')

Working directory: /home/ht/Documents/HT_GitHub/openmedallion/examples/sales_intelligence_demo


## Seed — generate source CSVs and write bronze Parquet

Creates 6 reps across 2 teams / 4 regions, 16 quarterly regional targets, and
60 closed-won transactions (Q1–Q3 2024).

In [2]:
!python seed.py

📄  Source CSVs written to data/source/
    reps.csv  — 6 rows × 4 cols
    targets.csv  — 16 rows × 3 cols
    transactions.csv  — 60 rows × 7 cols

📥   6 rows → data/bronze/reps.parquet
📥  16 rows → data/bronze/targets.parquet
📥  60 rows → data/bronze/transactions.parquet

✅  Bronze ready.

Next steps:
  medallion run sales_intel --layer silver
  medallion run sales_intel --layer gold
  python show_cerebrum.py


## Inspect source data

In [3]:
print('── reps ──')
print(pl.read_parquet('data/bronze/reps.parquet'))

── reps ──
shape: (6, 4)
┌────────┬────────────┬────────┬────────────┐
│ rep_id ┆ name       ┆ region ┆ team       │
│ ---    ┆ ---        ┆ ---    ┆ ---        │
│ i64    ┆ str        ┆ str    ┆ str        │
╞════════╪════════════╪════════╪════════════╡
│ 1      ┆ Alice Chen ┆ North  ┆ Enterprise │
│ 2      ┆ Bob Smith  ┆ South  ┆ SMB        │
│ 3      ┆ Carol Lee  ┆ East   ┆ Enterprise │
│ 4      ┆ David Kim  ┆ West   ┆ SMB        │
│ 5      ┆ Eve Brown  ┆ North  ┆ SMB        │
│ 6      ┆ Frank Wu   ┆ East   ┆ Enterprise │
└────────┴────────────┴────────┴────────────┘


In [4]:
print('── targets (first 8 rows) ──')
print(pl.read_parquet('data/bronze/targets.parquet').head(8))

── targets (first 8 rows) ──
shape: (8, 3)
┌────────┬─────────┬───────────────┐
│ region ┆ quarter ┆ target_amount │
│ ---    ┆ ---     ┆ ---           │
│ str    ┆ str     ┆ i64           │
╞════════╪═════════╪═══════════════╡
│ North  ┆ 2024-Q1 ┆ 80000         │
│ South  ┆ 2024-Q1 ┆ 60000         │
│ East   ┆ 2024-Q1 ┆ 90000         │
│ West   ┆ 2024-Q1 ┆ 70000         │
│ North  ┆ 2024-Q2 ┆ 85000         │
│ South  ┆ 2024-Q2 ┆ 65000         │
│ East   ┆ 2024-Q2 ┆ 95000         │
│ West   ┆ 2024-Q2 ┆ 75000         │
└────────┴─────────┴───────────────┘


In [5]:
print('── transactions (first 8 rows) ──')
print(pl.read_parquet('data/bronze/transactions.parquet').head(8))

── transactions (first 8 rows) ──
shape: (8, 7)
┌────────┬────────┬─────────────────┬────────┬───────┬────────────┬────────────┐
│ txn_id ┆ rep_id ┆ product         ┆ amount ┆ units ┆ txn_date   ┆ stage      │
│ ---    ┆ ---    ┆ ---             ┆ ---    ┆ ---   ┆ ---        ┆ ---        │
│ i64    ┆ i64    ┆ str             ┆ i64    ┆ i64   ┆ str        ┆ str        │
╞════════╪════════╪═════════════════╪════════╪═══════╪════════════╪════════════╡
│ 1      ┆ 1      ┆ SaaS Basic      ┆ 12000  ┆ 1     ┆ 2024-01-08 ┆ closed_won │
│ 2      ┆ 2      ┆ SaaS Pro        ┆ 28000  ┆ 1     ┆ 2024-01-11 ┆ closed_won │
│ 3      ┆ 3      ┆ SaaS Enterprise ┆ 85000  ┆ 1     ┆ 2024-01-15 ┆ closed_won │
│ 4      ┆ 4      ┆ SaaS Pro        ┆ 22000  ┆ 1     ┆ 2024-01-19 ┆ closed_won │
│ 5      ┆ 5      ┆ SaaS Basic      ┆ 9500   ┆ 2     ┆ 2024-01-22 ┆ closed_won │
│ 6      ┆ 6      ┆ SaaS Enterprise ┆ 75000  ┆ 1     ┆ 2024-01-26 ┆ closed_won │
│ 7      ┆ 1      ┆ SaaS Enterprise ┆ 92000  ┆ 1     ┆ 2024-0

---
## Run the full pipeline — Bronze → Silver → Gold

Silver casts columns then joins transactions × reps × targets into
`rep_performance.parquet` (a derived UDF table) with an `attainment_pct` column.
Gold aggregates that into three tables: `regional_summary`, `rep_leaderboard`,
`quarterly_trend`.

Pass `--no-explore` here to skip the `profile`/`walker` HTML reports (they require
`openmedallion[profile]` + `openmedallion[explore]`) — drop the flag to generate them.

In [6]:
!medallion run sales_intel --no-explore


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  medallion  ·  sales_intel  ·  bronze → silver → gold
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📋  [config] bronze: sales_intel/backend/bronze.yaml
📋  [config] silver: sales_intel/backend/silver.yaml
📋  [config] gold  : sales_intel/backend/gold.yaml

  ⏭️  bronze  skipped (existing files)
  ⏭️  silver  skipped (existing files)

── Gold ───────────────────────────────────────────────────
📊  [gold/sales_intel] regional_summary.parquet  (4 rows)
📊  [gold/sales_intel] rep_leaderboard.parquet  (6 rows)
⚙️   [gold]  udf add_quarter()  60 → 60 rows
📊  [gold/sales_intel] quarterly_trend.parquet  (3 rows)
🦆  [duckdb] 3 tables registered → data/gold/gold.duckdb  (mode: tables)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅  bronze → silver → gold complete.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



## Silver — rep_performance (derived join + attainment %)

In [7]:
silver_dir = Path('data/silver')
rep_perf = pl.read_parquet(silver_dir / 'rep_performance.parquet')
print(rep_perf.head(10))

shape: (10, 13)
┌────────┬────────┬──────────────┬─────────┬───┬────────┬────────────┬──────────────┬──────────────┐
│ txn_id ┆ rep_id ┆ product      ┆ amount  ┆ … ┆ region ┆ team       ┆ target_amoun ┆ attainment_p │
│ ---    ┆ ---    ┆ ---          ┆ ---     ┆   ┆ ---    ┆ ---        ┆ t            ┆ ct           │
│ i64    ┆ i64    ┆ str          ┆ f64     ┆   ┆ str    ┆ str        ┆ ---          ┆ ---          │
│        ┆        ┆              ┆         ┆   ┆        ┆            ┆ f64          ┆ f64          │
╞════════╪════════╪══════════════╪═════════╪═══╪════════╪════════════╪══════════════╪══════════════╡
│ 1      ┆ 1      ┆ SaaS Basic   ┆ 12000.0 ┆ … ┆ North  ┆ Enterprise ┆ 80000.0      ┆ 15.0         │
│ 2      ┆ 2      ┆ SaaS Pro     ┆ 28000.0 ┆ … ┆ South  ┆ SMB        ┆ 60000.0      ┆ 46.7         │
│ 3      ┆ 3      ┆ SaaS         ┆ 85000.0 ┆ … ┆ East   ┆ Enterprise ┆ 90000.0      ┆ 94.4         │
│        ┆        ┆ Enterprise   ┆         ┆   ┆        ┆            ┆     

## Gold — three aggregations

In [8]:
gold_dir = Path('data/gold/sales_intel')

print('── regional_summary ──')
print(pl.read_parquet(gold_dir / 'regional_summary.parquet'))

── regional_summary ──
shape: (4, 3)
┌────────┬───────────────┬───────────┐
│ region ┆ total_revenue ┆ num_deals │
│ ---    ┆ ---           ┆ ---       │
│ str    ┆ f64           ┆ u32       │
╞════════╪═══════════════╪═══════════╡
│ North  ┆ 744000.0      ┆ 20        │
│ South  ┆ 449000.0      ┆ 10        │
│ East   ┆ 993000.0      ┆ 20        │
│ West   ┆ 510000.0      ┆ 10        │
└────────┴───────────────┴───────────┘


In [9]:
print('── rep_leaderboard ──')
print(pl.read_parquet(gold_dir / 'rep_leaderboard.parquet').sort('total_revenue', descending=True))

── rep_leaderboard ──
shape: (6, 6)
┌────────┬────────────┬────────┬────────────┬───────────────┬───────────┐
│ rep_id ┆ name       ┆ region ┆ team       ┆ total_revenue ┆ num_deals │
│ ---    ┆ ---        ┆ ---    ┆ ---        ┆ ---           ┆ ---       │
│ i64    ┆ str        ┆ str    ┆ str        ┆ f64           ┆ u32       │
╞════════╪════════════╪════════╪════════════╪═══════════════╪═══════════╡
│ 6      ┆ Frank Wu   ┆ East   ┆ Enterprise ┆ 534500.0      ┆ 10        │
│ 4      ┆ David Kim  ┆ West   ┆ SMB        ┆ 510000.0      ┆ 10        │
│ 3      ┆ Carol Lee  ┆ East   ┆ Enterprise ┆ 458500.0      ┆ 10        │
│ 2      ┆ Bob Smith  ┆ South  ┆ SMB        ┆ 449000.0      ┆ 10        │
│ 1      ┆ Alice Chen ┆ North  ┆ Enterprise ┆ 411000.0      ┆ 10        │
│ 5      ┆ Eve Brown  ┆ North  ┆ SMB        ┆ 333000.0      ┆ 10        │
└────────┴────────────┴────────┴────────────┴───────────────┴───────────┘


In [10]:
print('── quarterly_trend ──')
print(pl.read_parquet(gold_dir / 'quarterly_trend.parquet').sort('quarter'))

── quarterly_trend ──
shape: (3, 3)
┌─────────┬───────────────────┬───────────┐
│ quarter ┆ quarterly_revenue ┆ num_deals │
│ ---     ┆ ---               ┆ ---       │
│ str     ┆ f64               ┆ u32       │
╞═════════╪═══════════════════╪═══════════╡
│ 2024-Q1 ┆ 1.007e6           ┆ 24        │
│ 2024-Q2 ┆ 1.171e6           ┆ 26        │
│ 2024-Q3 ┆ 518000.0          ┆ 10        │
└─────────┴───────────────────┴───────────┘


---
## cerebrum — natural-language queries (no Ollama required)

`show_cerebrum.py` walks through the schema context the LLM receives, a full prompt
preview, three live DuckDB queries executed directly against silver Parquet, and a
mock recommender call — all without a running LLM server.

In [11]:
!python show_cerebrum.py


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  cerebrum demo  ·  sales_intel  ·  offline mode (no Ollama needed)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


── 1. Schema context (what the LLM receives) ──────────────────────
TABLE rep_performance (
  txn_id                       BIGINT,
  rep_id                       BIGINT,
  product                      VARCHAR,
  amount                       DOUBLE,
  units                        BIGINT,
  txn_date                     VARCHAR,
  stage                        VARCHAR,
  quarter                      VARCHAR,
  name                         VARCHAR,
  region                       VARCHAR,
  team                         VARCHAR,
  target_amount                DOUBLE,
  attainment_pct               DOUBLE
)

TABLE reps (
  rep_id                       BIGINT,
  name                         VARCHAR,
  region                       VARCHAR,
  team                         VARCHAR
)

TABLE targets

### Try it with a real LLM

Requires `pip install "openmedallion[cerebrum]"` and a running Ollama server
(`ollama serve`), or set `MEDALLION_LLM_PROVIDER`/`MEDALLION_LLM_API_KEY` for a
cloud provider (see the main [README](../../README.md#provider-options)).

In [ ]:
# Uncomment to run against a live LLM:
# !medallion query sales_intel "Which rep had the highest revenue in Q2?"
# !medallion query sales_intel "Headcount by team and revenue by region" --decompose
# !medallion query sales_intel "Show me the good ones" --detect-ambiguity
# !medallion query sales_intel "Which rep had the highest revenue in Q2?" --user alice

---
## RAG accuracy — metadata, relationships, examples, confidence-gated fallback

This project ships with curated `sales_intel/metadata.yaml`, `sales_intel/relationships.yaml`,
and `sales_intel/examples/synthetic.jsonl`. `show_rag_workflow.py` demonstrates the full
retrieval pipeline **without Ollama or ChromaDB installed**, using an injected
deterministic word-overlap embedding function in place of a real one.

It shows: curated knowledge, dynamic few-shot retrieval, and both sides of the
confidence gate (a question that clears the 0.7 threshold vs. one that doesn't and
falls back to a raw-schema search over every silver table).

In [ ]:
!python show_rag_workflow.py

### Regenerate / review curated knowledge with a real LLM

```bash
medallion metadata generate      sales_intel     # draft descriptions for any new tables
medallion metadata approve       sales_intel     # human review, one table at a time
medallion metadata refresh       sales_intel     # detect schema drift, re-draft affected tables
medallion metadata refresh       sales_intel --check   # CI gate: exit 1 if schema drifted

medallion relationships generate sales_intel     # no LLM call — pure pattern matching
medallion relationships approve  sales_intel
medallion relationships erd      sales_intel     # Mermaid ER diagram → relationships_erd.md

medallion examples generate      sales_intel --count 15
medallion examples approve       sales_intel

# After closing a cortex session (End Session button / 5min idle / tab close —
# session close is what promotes rated turns, not the thumbs click itself)
medallion examples harvest sales_intel           # promote thumbs-up into synthetic.jsonl
medallion examples review  sales_intel           # list thumbs-down failures
medallion examples eval    sales_intel           # regression check: did curation help or hurt?
```

Every question — from cortex or `medallion query` — is logged to
`sales_intel/chat_history/<user>.jsonl` (a personal audit trail, never fed back
into the LLM prompt). A 👍/👎 click sets that turn's `accepted` field in place
(three-state — never inferred from behavior); closing the session is what rolls
rated turns into `harvested.jsonl`/`failures.jsonl` above.

---
## Go live — neuron server + cortex chat UI

```bash
# Terminal 1
medallion ask sales_intel
# → http://localhost:8000/docs (Swagger UI), /health, /query, /feedback

# Terminal 2
medallion cortex sales_intel
# → open http://localhost:8050 — chat with 👍/👎 feedback, table, dashboard tabs
```

## Things to Try

- **Ask a new question**: try one of the sample questions in `README.md` (`Which team —
  Enterprise or SMB — closed more deals?`, `Who is closest to hitting their quarterly
  target?`) against a real LLM via `medallion query`.
- **Add a new rep or transaction**: edit `seed.py`, delete `data/`, and re-run this
  notebook from the top.
- **Approve more tables**: `medallion metadata approve sales_intel` to bring an
  additional table into schema pruning's approved corpus.
- **Trigger the confidence-gate fallback**: ask something off-topic and watch
  `show_rag_workflow.py`'s output fall back to raw-schema search.
- **Explore visually**: re-run `medallion run sales_intel` without `--no-explore` and
  open the generated `profile`/`walker` HTML reports under `data/<layer>/add-ons/`.